In [ ]:
import sys
import os
import random
import pandas as pd
import librosa
import torchaudio
import torch
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_DIR))
print(PROJECT_DIR)

In [ ]:
from src.core.config import settings, update_path_settings
from src.domain.pipelines.annotations import load_annotation_file, get_annotation_file

update_path_settings(project_dir=PROJECT_DIR)
SECRETS_DIR = settings.SECRETS_DIR or Path("secrets")
DATA_ZIP_DIR = settings.DATA_ZIP_DIR or Path("data") / "zip"
DATA_RAW_DIR = settings.DATA_RAW_DIR or Path("data") / "raw"
DATA_PREPROCESSED_DIR = settings.DATA_PREPROCESSED_DIR or Path("data") / "preprocessed"

In [ ]:
def load_audio(
    audio_file: Path, sample_rate: int | float | None = None
) -> tuple[npt.NDArray[np.float32], int | float]:
    return librosa.load(audio_file, sr=sample_rate)


def clip_audio(
    waveform: npt.NDArray[np.float32] | torch.Tensor,
    limit_sec: float,
    offset_sec: float,
    sample_rate: int | float,
) -> npt.NDArray[np.float32] | torch.Tensor:
    start_sample = int(offset_sec * sample_rate)
    end_sample = int((offset_sec + limit_sec) * sample_rate)
    return waveform[start_sample:end_sample]

In [ ]:
random_dir = random.choice(
    [d for d in os.listdir(DATA_RAW_DIR) if (DATA_RAW_DIR / d).is_dir()]
)
SAMPLE_DIR = DATA_RAW_DIR / random_dir
ANNOTATIONS_DIR = DATA_PREPROCESSED_DIR / random_dir
SAMPLE_DIR, ANNOTATIONS_DIR

In [ ]:
from torchaudio.transforms import Spectrogram

NFFT = 2048
HOP_LENGTH = 256
SAMPLE_RATE = 32000
spec = Spectrogram(n_fft=NFFT, hop_length=HOP_LENGTH)

records_files: list[Path] = [
    SAMPLE_DIR / wav_file
    for wav_file in os.listdir(SAMPLE_DIR)
    if wav_file.endswith(".wav")
]
sample_record: Path = random.choice(records_files)
sample_annotation: Path | None = get_annotation_file(ANNOTATIONS_DIR, sample_record)

annotations: pd.DataFrame | None = (
    load_annotation_file(sample_annotation) if sample_annotation else None
)

audio, sr = load_audio(sample_record, sample_rate=SAMPLE_RATE)
clip_duration_sec = 5.0
offset_sec = 0.0
cliped = clip_audio(audio, clip_duration_sec, offset_sec, sr)

spectrogram = spec(torch.from_numpy(cliped)).numpy()
plt.figure(figsize=(16, 8))
plt.imshow(10 * np.log10(spectrogram + 1e-10), aspect="auto", origin="lower")
plt.title(f"Spectrogram {sample_record.name}")
plt.show()

In [ ]:
print(f"Audio Shape: {audio.shape}, Sample Rate: {sr}")
print(
    f"Spectrogram Shape: {spectrogram.shape} == {(NFFT // 2 + 1, int(np.ceil(len(cliped) / HOP_LENGTH)))}"
)

annotations

In [ ]:
from collections.abc import Callable

COORDINATE = tuple[int, int, int, int, str, str]  # (x1, y1, x2, y2, species, call_type)
YOLO_LABEL = tuple[int, float, float, float, float]  # (class_id, xc, yc, w, h)


def get_global_cords(
    sr: int | float, nfft: int, hop_length: int, annotations: pd.DataFrame
) -> list[COORDINATE]:
    global_cords = []
    for _, row in annotations.iterrows():
        start_time = row["begin_time"]
        end_time = row["end_time"]
        low_freq = row["low_freq"]
        high_freq = row["high_freq"]

        x1 = int(start_time * sr / hop_length)
        x2 = int(end_time * sr / hop_length)
        y1 = int(low_freq * nfft / sr)
        y2 = int(high_freq * nfft / sr)

        species = row["specie"]
        call_type = row["call_type"]

        global_cords.append((x1, y1, x2, y2, species, call_type))
    return global_cords


def global_cords_to_yolo(
    global_cords: list[COORDINATE],
    img_w: int,
    img_h: int,
    class_to_id: Callable[[str, str], int],
) -> list[YOLO_LABEL]:
    yolo_labels: list[YOLO_LABEL] = []
    for x1, y1, x2, y2, species, call_type in global_cords:

        x1 = max(0, min(x1, img_w - 1))
        x2 = max(0, min(x2, img_w - 1))
        y1 = max(0, min(y1, img_h - 1))
        y2 = max(0, min(y2, img_h - 1))
        if x2 <= x1 or y2 <= y1:
            continue

        xc = ((x1 + x2) / 2) / img_w
        yc = ((y1 + y2) / 2) / img_h
        w = (x2 - x1) / img_w
        h = (y2 - y1) / img_h

        yolo_labels.append((class_to_id(species, call_type), xc, yc, w, h))
    return yolo_labels

In [ ]:
global_cords = (
    get_global_cords(sr, NFFT, HOP_LENGTH, annotations)
    if annotations is not None
    else []
)
global_cords

In [ ]:
from src.domain.utils.species import get_species_id


yolo_cords = global_cords_to_yolo(
    global_cords=global_cords,
    img_w=spectrogram.shape[1],
    img_h=spectrogram.shape[0],
    class_to_id=lambda species, _: get_species_id(species),
)
yolo_cords

In [ ]:
from matplotlib.patches import Rectangle

spectrogram = spec(torch.from_numpy(audio)).numpy()
plt.figure(figsize=(16, 8))
plt.imshow(10 * np.log10(spectrogram + 1e-10), aspect="auto", origin="lower")
plt.title(f"Spectrogram {sample_record.name}")
for x1, y1, x2, y2, species, call_type in global_cords:
    plt.gca().add_patch(
        Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )
    )
plt.show()

In [ ]:
from matplotlib.patches import Rectangle

spectrogram_full = spec(torch.from_numpy(audio)).numpy()
img_h, img_w = spectrogram_full.shape

plt.figure(figsize=(16, 8))
plt.imshow(10 * np.log10(spectrogram_full + 1e-10), aspect="auto", origin="lower")
plt.title(f"YOLO boxes sobre espectrograma - {sample_record.name}")

for class_id, xc, yc, w, h in yolo_cords:
    box_w = w * img_w
    box_h = h * img_h
    x1 = (xc * img_w) - (box_w / 2)
    y1 = (yc * img_h) - (box_h / 2)

    plt.gca().add_patch(
        Rectangle(
            (x1, y1),
            box_w,
            box_h,
            edgecolor="lime",
            facecolor="none",
            linewidth=2,
        )
    )
    plt.text(x1, y1, str(class_id), color="white", fontsize=8, backgroundcolor="black")

plt.show()